# Data Enrichment: Building the Combined European Film Dataset

This notebook documents how we enriched our original European film dataset (2,000 films from IMDb) with three additional data sources to enable deeper analysis of audience reception and commercial performance.

## Data Sources

| Source | What it adds | Coverage |
|--------|-------------|----------|
| **Original dataset** | IMDb metadata (title, genre, country, cast, plot, keywords, ratings) | 2,000 films |
| **TMDb API** | Budget, revenue, popularity, tagline, production countries | 89.5% found, 53% have financials |
| **IMDb GraphQL API** | Per-star rating distributions (vote counts for each score 1–10) | 99.95% coverage |
| **MovieLens 32M** | User-level ratings, standard deviation, user-contributed tags | 79.7% found, 72.5% have tags |

## Why these sources?

Our original dataset had average IMDb ratings but no insight into **how** people rated films — are ratings concentrated in the middle, or split between love and hate? We also had no financial data to connect audience reception to commercial outcomes.

- **IMDb rating distributions** let us measure **polarization** — films that divide audiences
- **TMDb budget/revenue** lets us analyze **commercial performance and ROI**
- **MovieLens** gives us a second rating platform with **user-level granularity and tags**

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 70)

## 1. Load the Raw Sources

In [ ]:
# Original dataset
main = pd.read_excel("../03-data/European_data_2000.xlsx")
print(f"Original dataset: {len(main)} films, {len(main.columns)} columns")
main.head(3)

In [ ]:
# TMDb enrichment (budget, revenue, popularity)
tmdb = pd.read_csv("../03-data/raw/tmdb_enrichment.csv")
print(f"TMDb data: {len(tmdb)} films")
print(f"  Found on TMDb: {tmdb['tmdb_id'].notna().sum()} ({tmdb['tmdb_id'].notna().mean():.1%})")
print(f"  Has budget: {(tmdb['budget'].fillna(0) > 0).sum()}")
print(f"  Has revenue: {(tmdb['revenue'].fillna(0) > 0).sum()}")
tmdb[tmdb['budget'] > 0].head(3)

In [ ]:
# IMDb rating distributions (per-star vote counts)
imdb_hist = pd.read_csv("../03-data/raw/imdb_rating_distributions.csv")
print(f"IMDb histograms: {len(imdb_hist)} films")
print(f"  Has data: {imdb_hist['totalVotes'].notna().sum()} ({imdb_hist['totalVotes'].notna().mean():.1%})")
imdb_hist.head(3)

In [ ]:
# MovieLens enrichment (user-level ratings + tags)
ml = pd.read_csv("../03-data/raw/movielens_enrichment.csv")
print(f"MovieLens data: {len(ml)} films")
print(f"  Has tags: {ml['ml_tags'].notna().sum()} ({ml['ml_tags'].notna().mean():.1%})")
print(f"  Total ratings: {ml['ml_rating_count'].sum():,.0f}")
ml.head(3)

## 2. Merge into Combined Dataset

All sources are joined on `titleId` (IMDb identifier). We use left joins to keep all 2,000 films even if a source doesn't cover them.

In [ ]:
df = main.merge(tmdb, on="titleId", how="left")
df = df.merge(imdb_hist, on="titleId", how="left")
df = df.merge(ml, on="titleId", how="left")

print(f"Combined dataset: {len(df)} films, {len(df.columns)} columns")
df.info()

## 3. Coverage Overview

How much data do we actually have across sources?

In [ ]:
coverage = pd.DataFrame({
    "Source": ["Original (IMDb metadata)", "TMDb (found)", "TMDb (has budget or revenue)",
              "IMDb histograms", "MovieLens (ratings)", "MovieLens (tags)",
              "All 3 enrichment sources"],
    "Films": [
        len(df),
        df["tmdb_id"].notna().sum(),
        ((df["budget"].fillna(0) > 0) | (df["revenue"].fillna(0) > 0)).sum(),
        df["totalVotes"].notna().sum(),
        df["ml_rating_count"].notna().sum(),
        df["ml_tags"].notna().sum(),
        ((df["tmdb_id"].notna()) & (df["totalVotes"].notna()) & (df["ml_rating_count"].notna())).sum(),
    ]
})
coverage["Coverage"] = (coverage["Films"] / len(df) * 100).round(1).astype(str) + "%"
coverage

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(coverage["Source"], coverage["Films"], color="steelblue")
ax.set_xlabel("Number of films")
ax.set_title("Data Coverage by Source (out of 2,000 films)")
ax.axvline(x=2000, color="gray", linestyle="--", alpha=0.5)
for bar, val in zip(bars, coverage["Films"]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

## 4. New Columns Explained

### From TMDb
| Column | Description |
|--------|-------------|
| `budget` | Production budget in USD |
| `revenue` | Worldwide box office revenue in USD |
| `tmdb_popularity` | TMDb popularity score (views, watchlists, etc.) |
| `tmdb_vote_average` | TMDb user rating (0-10) |
| `tmdb_vote_count` | Number of TMDb user votes |
| `original_language` | ISO language code |
| `tagline` | Marketing tagline |
| `production_countries` | ISO country codes |

### From IMDb Rating Distributions
| Column | Description |
|--------|-------------|
| `rating_1` to `rating_10` | Number of IMDb users who gave each score |
| `aggregateRating` | Weighted average (same as `imdbRating`) |
| `totalVotes` | Total votes (same as `numberOfVotes`) |

### From MovieLens
| Column | Description |
|--------|-------------|
| `ml_rating_count` | Number of MovieLens ratings |
| `ml_rating_mean` | Average MovieLens rating (0.5–5.0 scale) |
| `ml_rating_std` | Standard deviation of ratings |
| `ml_rating_median` | Median rating |
| `ml_rating_0_5` to `ml_rating_5_0` | Ratings per half-star bucket |
| `ml_tags` | User-contributed tags (comma-separated) |

## 5. Quick Sanity Checks

In [ ]:
# Do IMDb and TMDb ratings agree?
has_both = df[df["tmdb_vote_average"].notna() & df["imdbRating"].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(has_both["imdbRating"], has_both["tmdb_vote_average"], alpha=0.3, s=10)
axes[0].plot([0, 10], [0, 10], "r--", alpha=0.5)
axes[0].set_xlabel("IMDb Rating")
axes[0].set_ylabel("TMDb Rating")
axes[0].set_title("IMDb vs TMDb Ratings")

# Do IMDb and MovieLens ratings agree? (different scales: IMDb 1-10, ML 0.5-5)
has_ml = df[df["ml_rating_mean"].notna()].copy()
axes[1].scatter(has_ml["imdbRating"], has_ml["ml_rating_mean"] * 2, alpha=0.3, s=10)
axes[1].plot([0, 10], [0, 10], "r--", alpha=0.5)
axes[1].set_xlabel("IMDb Rating")
axes[1].set_ylabel("MovieLens Rating (scaled to 0-10)")
axes[1].set_title("IMDb vs MovieLens Ratings")

plt.tight_layout()
plt.show()

In [ ]:
# Budget and revenue distributions
has_fin = df[(df["budget"].fillna(0) > 0) | (df["revenue"].fillna(0) > 0)].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

has_fin[has_fin["budget"] > 0]["budget"].apply(lambda x: x / 1e6).hist(
    bins=50, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_xlabel("Budget (millions USD)")
axes[0].set_ylabel("Number of films")
axes[0].set_title(f"Budget Distribution (n={int((has_fin['budget'] > 0).sum())})")

has_fin[has_fin["revenue"] > 0]["revenue"].apply(lambda x: x / 1e6).hist(
    bins=50, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_xlabel("Revenue (millions USD)")
axes[1].set_ylabel("Number of films")
axes[1].set_title(f"Revenue Distribution (n={int((has_fin['revenue'] > 0).sum())})")

plt.tight_layout()
plt.show()

In [ ]:
# Example: rating distribution for a polarizing film
rating_cols = [f"rating_{i}" for i in range(1, 11)]

# Find most polarizing film with decent vote count
subset = df[df["totalVotes"] > 1000].copy()
subset["pct_extreme"] = (subset["rating_1"].fillna(0) + subset["rating_10"].fillna(0)) / subset["totalVotes"]
most_polar = subset.nlargest(1, "pct_extreme").iloc[0]

# Also pick a consensus film for comparison
subset["pct_middle"] = subset[[f"rating_{i}" for i in range(5, 9)]].sum(axis=1) / subset["totalVotes"]
most_consensus = subset.nlargest(1, "pct_middle").iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, film, label in [(axes[0], most_polar, "Most Polarizing"), (axes[1], most_consensus, "Most Consensus")]:
    votes = [film[f"rating_{i}"] for i in range(1, 11)]
    ax.bar(range(1, 11), votes, color="steelblue", edgecolor="white")
    ax.set_xlabel("Rating")
    ax.set_ylabel("Vote count")
    title = film["englishTitle"] if pd.notna(film["englishTitle"]) else film["originalTitle"]
    ax.set_title(f"{label}: {title}\n(avg: {film['imdbRating']}, votes: {int(film['totalVotes']):,})")
    ax.set_xticks(range(1, 11))

plt.tight_layout()
plt.show()

## 6. Sample Tags from MovieLens

MovieLens user tags give us thematic descriptors that go beyond genre labels.

In [ ]:
# Show some example tags
tagged = df[df["ml_tags"].notna()][["originalTitle", "genres", "ml_tags"]].head(10)
for _, row in tagged.iterrows():
    tags = row["ml_tags"][:100] + ("..." if len(str(row["ml_tags"])) > 100 else "")
    print(f"{row['originalTitle']}")
    print(f"  Genres: {row['genres']}")
    print(f"  Tags:   {tags}")
    print()

## 7. Save Combined Dataset

In [ ]:
df.to_csv("../03-data/european_films_enriched.csv", index=False)
print(f"Saved: {len(df)} films, {len(df.columns)} columns")
print(f"File: 03-data/european_films_enriched.csv")

## Summary

We started with 2,000 European films and 23 columns of IMDb metadata. After enrichment, we have **61 columns** including:

- **Financial data** (budget + revenue) for ~1,060 films from TMDb
- **Rating distributions** (per-star vote counts) for 1,999 films from IMDb's GraphQL API
- **User-level rating stats and tags** for ~1,593 films from MovieLens 32M

### Scripts used
| Script | Source | API key needed? |
|--------|--------|----------------|
| `src/fetch_tmdb.py` | TMDb API | Yes (free at themoviedb.org) |
| `src/fetch_imdb_ratings.py` | IMDb GraphQL API | No |
| `src/fetch_movielens.py` | MovieLens 32M dataset | No (download zip) |

### Raw data files
All raw enrichment files are stored in `03-data/raw/`:
- `tmdb_enrichment.csv`
- `imdb_rating_distributions.csv`
- `movielens_enrichment.csv`

The combined dataset is at `03-data/european_films_enriched.csv`.

### Limitations
- **TMDb financial data is community-contributed** — coverage skews toward well-known films. ~47% of our films have no budget or revenue data.
- **MovieLens users are not representative** — they skew younger, English-speaking, and more film-literate than general audiences.
- **IMDb rating distributions are aggregated** — we know how many people gave a 10, but not who they are or their other ratings.
- **Currency/inflation** — TMDb budget and revenue figures are nominal USD with no adjustment for inflation or purchasing power parity.